Content-Based Recommendation

1. Preparing data

In [1]:
import pandas as pd
import os
import pickle

# Step 1: Load data
movies = pd.read_csv('../data/movies.csv')
ratings = pd.read_csv('../data/ratings.csv')

# Step 2: Create content_df
content_df = movies[['title', 'genres']].copy()
content_df.dropna(inplace=True)

content_df.head()

,title,genres
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,Jumanji (1995),Adventure|Children|Fantasy
2,Grumpier Old Men (1995),Comedy|Romance
3,Waiting to Exhale (1995),Comedy|Drama|Romance
4,Father of the Bride Part II (1995),Comedy


Clean titles

In [2]:
# Remove year → "Toy Story (1995)" → "Toy Story"
content_df['clean_title'] = content_df['title'].str.replace(r'\(\d{4}\)', '', regex=True).str.strip()

2. Genres cleaning

In [3]:
# Convert "Action|Adventure" → "Action Adventure"
content_df['genres'] = content_df['genres'].str.replace('|', ' ', regex=False)

# Optional cleanup
content_df = content_df[content_df['genres'] != '(no genres listed)']

3. TF-IDF Vectorization

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Convert genres into TF-IDF vectors (numerical representation)
tfidf = TfidfVectorizer()

tfidf_matrix = tfidf.fit_transform(content_df['genres'])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (9708, 21)


4. Cosine Similarity

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute similarity between all movies
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Cosine Similarity Matrix Shape:", cosine_sim.shape)

Cosine Similarity Matrix Shape: (9708, 9708)


5. Creating Title Index Mapping

In [6]:
# Using clean_title instead of title
indices = pd.Series(content_df.index, index=content_df['clean_title']).drop_duplicates()

6. Recommendation Function (Final Version)

In [7]:
def recommend_movies(title, top_n=10):
    
    # Better matching
    title = title.lower().strip()
    matches = [t for t in indices.index if title in t.lower()]    
    if not matches:
        return "Movie not found. Try a different name."
    
    matched_title = matches[0]
    idx = indices[matched_title]
    
    # Get similarity scores 
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort movies based on similarity
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Skip the first one (itself)
    sim_scores = sim_scores[1:top_n+1]
    
    # Get movie indices and similarity scores
    movie_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    # Create result DataFrame
    results = pd.DataFrame({
        'title': content_df['title'].iloc[movie_indices],
        'similarity_score': scores
    })
    
    print(f"\nBecause you searched: {matched_title}")
    
    return results.reset_index(drop=True)

7. Testing the Model

In [8]:
# Check available movie titles
content_df['title'].sample(10)

163                        Showgirls (1995)
5136                    Safety Last! (1923)
986                          Ben-Hur (1959)
5369           Lightning in a Bottle (2004)
3060             Ernest Goes to Camp (1987)
7752                          Lifted (2006)
3336           Land Before Time, The (1988)
8370             Someone Marry Barry (2014)
2347                      Awakenings (1990)
9662    Craig Ferguson: Tickle Fight (2017)
Name: title, dtype: object

In [9]:
recommend_movies('Toy Story')


Because you searched: Toy Story


,title,similarity_score
0,Antz (1998),1.0
1,Toy Story 2 (1999),1.0
2,"Adventures of Rocky and Bullwinkle, The (2000)",1.0
3,"Emperor's New Groove, The (2000)",1.0
4,"Monsters, Inc. (2001)",1.0
5,"Wild, The (2006)",1.0
6,Shrek the Third (2007),1.0
7,"Tale of Despereaux, The (2008)",1.0
8,Asterix and the Vikings (Astérix et les Viking...,1.0
9,Turbo (2013),1.0


Creating folders for saving

In [10]:
os.makedirs('../data', exist_ok=True)
os.makedirs('../models', exist_ok=True)

Saving cleaned data

In [11]:
content_df.to_csv('../data/content_data.csv', index=False)

Saving models

In [12]:
with open('../models/tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

with open('../models/cosine_sim.pkl', 'wb') as f:
    pickle.dump(cosine_sim, f)

with open('../models/indices.pkl', 'wb') as f:
    pickle.dump(indices, f)